# 🧩 Golden Dataset Fusion Logic (Detailed Explanation)

This section documents the fusion logic for each column in the **golden fused dataset**.  
Each rule explains *how* and *why* conflicting or missing values are resolved across datasets  
(Amazon, Goodreads, Metabooks).

---

### 1️⃣ `title`
**Goal:** Use the cleanest, canonical title (remove edition/subtitle noise).

**Fusion logic:**
- Collect all non-null titles from all sources.
- Clean:
  - Strip trailing parentheses/brackets → `"Book (Revised Edition)" → "Book"`.
  - Remove subtitles → `"Book: A Novel" → "Book"`.
- Deduplicate (case-insensitive).
- Choose **shortest** cleaned version; ties → fewer tokens → alphabetical.

**Rationale:**  
Shorter titles represent canonical works, while longer ones contain edition info.

---

### 2️⃣ `author`
**Goal:** Keep the main author, remove editors or collaborators.

**Fusion logic:**
- Take text before first comma/parenthesis → `"Author, Editor"` → `"Author"`.
- Normalize spacing.
- If multiple unique names, pick **longest** (most complete form).

**Rationale:**  
Ensures only the principal author remains consistent across datasets.

---

### 3️⃣ `publisher`
**Goal:** Unify publisher names that vary by imprint or suffix.

**Fusion logic:**
- Lowercase + remove tokens like `publishers`, `press`, `books`, `group`, etc.
- Count normalized forms.
- Pick **most frequent** root; ties → **longest** name.

**Rationale:**  
Removes corporate/legal noise and standardizes imprints under one label.

---

### 4️⃣ `publish_year`
**Goal:** Select the earliest valid publication year.

**Fusion logic:**
- Convert to numeric and drop invalids (<1400, >2100).
- Take **minimum non-null** year.

**Rationale:**  
Older date represents original publication; newer ones are reprints.

---

### 5️⃣ `rating`
**Goal:** Produce a unified average user rating (from Goodreads & Metabooks).

**Fusion logic:**
1. Extract `(rating, numratings)` from Goodreads and Metabooks.
2. If both exist:
   - Compute relative difference:
     ```
     diff = |n1 - n2| / max(n1, n2)
     ```
   - If `diff ≤ 0.15` → use **weighted average**:
     ```
     fused = (r1*n1 + r2*n2) / (n1 + n2)
     ```
   - Else → take rating from source with **more votes**.
3. If only one exists → use that one.

**Rationale:**  
Weighted averaging improves precision when vote counts are similar;  
dominant-source rule ensures reliability when one source is stronger.

---

### 6️⃣ `numratings`
**Goal:** Consistent total number of ratings (aligned with fused rating).

**Fusion logic:**
- Same proximity rule (`≤15%` difference).
- If close → use **sum (n1 + n2)**.
- Else → **max(n1, n2)** (dominant source).
- Fallback → single-source value.

**Rationale:**  
Combines user bases when comparable; otherwise trusts the larger population.

---

### 7️⃣ `genres`
**Goal:** Merge all distinct genres into a unified set.

**Fusion logic:**
- Use Goodreads + Metabooks only.
- Parse stringified lists like `"['Fiction','Drama']"`,  
  or split delimited strings `"Fiction, Drama"`.
- Normalize: lowercase + strip + title-case.
- Take **union of unique genres**, sorted alphabetically.
- Return as comma-separated string.

**Rationale:**  
Preserves all genre information, removes duplicates, standardizes format.

---

### 8️⃣ `page_count`
**Goal:** Represent canonical number of pages for the main edition.

**Fusion logic:**
- Use Goodreads + Metabooks.
- Drop invalid (<20 pages).
- If both exist:
  - If difference ≤5% → **mean**.
  - Else → **prefer Goodreads** (default).
- If one present → use it.

**Rationale:**  
Handles small edition variances; Goodreads usually more standardized.

---

### 9️⃣ `price`
**Goal:** Use the most reliable retail-style price.

**Fusion logic:**
- Convert to numeric, drop non-positive values.
- Prefer **Metabooks**, else **Goodreads**, else `None`.
- Round to 2 decimals.

**Rationale:**  
Amazon lacks prices; Metabooks derived from Amazon → most reliable fallback.

---

### 🔟 `language`
**Goal:** Evaluate and handle language consistency across datasets.

**Analysis:**
- Around 70–80 unique language values existed overall (e.g., English, French, German, Spanish, etc.).
- For each ISBN, all sources that provided a language agreed — there were **no inter-dataset conflicts**.
- Most discrepancies came from **missing values (`NaN`)**, not disagreements.

**Decision:**
- Since the column did not vary between datasets for any given ISBN and contained many nulls,
  it provided no additional information for fusion.
- Therefore, the `language` column was **dropped** from the final golden dataset.

**Rationale:**  
Retaining `language` would have added redundant and mostly missing data.  
Dropping it simplified the integrated schema without any loss of information or fidelity.

---

### ⚙️ Filtering Step
Before fusion:
```python
# Keep only ISBNs appearing in at least two datasets
long = long[long.groupby("isbn_clean")["source"].transform("nunique") >= 2]


In [19]:
import pandas as pd
amazon=pd.read_parquet("parquet/amazon_new.parquet")
goodreads=pd.read_parquet("parquet/goodreads_new.parquet")
# We are dropping columns that are not needed for data fusion. These two columns only exist in goodreads dataset
metabooks=pd.read_parquet("parquet/metabooks_new.parquet")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)



In [3]:
display(amazon.head(1).style.set_caption("Amazon"))
display(goodreads.head(1).style.set_caption("Goodreads"))
display(metabooks.head(1).style.set_caption("Metabooks"))


,id,title,author,publish_year,publisher,isbn_clean
0,amazon_1,classical mythology,mark p o morford,2002,Oxford University Press,0195153448


,id,title,author,rating,numratings,language,genres,bookformat,edition,page_count,publisher,publish_year,price,isbn_clean
0,goodreads_1,the hunger games,suzanne collins,4.330000,6376780,English,"['Young Adult', 'Fiction', 'Dystopia', 'Fantasy', 'Science Fiction', 'Romance', 'Adventure', 'Teen', 'Post Apocalyptic', 'Action']",Hardcover,First Edition,374,Scholastic Press,2008,5.090000,0439023483


,id,title,author,rating,numratings,language,genres,publisher,page_count,price,publish_year,isbn_clean
0,metabooks_1,alvis the story of the red triangle,kenneth day,4.100000,3,English,"['Engineering', 'Transportation', 'Automotive']",Haynes Pubns,400,112.300003,,1844255247


In [4]:
common_isbns = list(
    set(amazon['isbn_clean'])
    & set(goodreads['isbn_clean'])
    & set(metabooks['isbn_clean'])
)

In [ ]:
import pandas as pd, numpy as np, re, ast

# Amazon is missing these (in your setup):
AMAZON_MISSING = ["rating","numratings","language","genres","page_count","price"]

# Union schema we care about
BASE_COLS = [
    "id","title","author","publisher","publish_year",
    "rating","numratings","genres","page_count","price","isbn_clean"
]

def _add_missing_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def build_longform(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Normalize inputs to the same schema, add source label, and stack them."""
    amz = _add_missing_cols(amazon.copy(), AMAZON_MISSING)
    gr  = goodreads.copy()
    mb  = metabooks.copy()

    amz["source"] = "amazon"
    gr["source"]  = "goodreads"
    mb["source"]  = "metabooks"

    def keep_cols(df): 
        return df[[c for c in BASE_COLS if c in df.columns] + ["source"]]
    
    long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)

    # normalize empties to NaN
    for c in ["title","author","publisher","genres","isbn_clean","id"]:
        if c in long.columns: 
            long[c] = long[c].replace({"": np.nan})
    return long


# ---------- title (Metabooks strategy: take the LONGEST cleaned title) ----------
def fuse_title(g: pd.DataFrame):
    titles = g["title"].dropna().astype(str).tolist()
    if not titles: 
        return None
    def clean_title(t):
        t = t.strip()
        t = re.sub(r"\s*[\(\[].*?[\)\]]\s*$", "", t)  # drop trailing (...) or [...]
        t = re.split(r"\s*[:\-]\s+", t)[0]            # cut at ":" or "-"
        return t.strip()
    pairs, dedup = [(t, clean_title(t)) for t in titles], {}
    for orig, c in pairs:
        key = c.lower()
        if key not in dedup: 
            dedup[key] = orig
    # pick the LONGEST cleaned key; ties -> more tokens -> reverse alphabetical for determinism
    best_key = max(dedup.keys(), key=lambda k: (len(k), len(k.split()), k))
    return dedup[best_key]


# ---------- author (compare Amazon + Metabooks only; take LONGEST; fallback Goodreads) ----------
def fuse_author(g: pd.DataFrame):
    # candidates only from amazon + metabooks
    sub = g[g["source"].isin(["amazon","metabooks"])]
    vals = sub["author"].dropna().astype(str).tolist()
    if not vals:
        # fallback to Goodreads if neither amazon nor metabooks present
        gr = g[g["source"].eq("goodreads")]["author"].dropna().astype(str)
        return gr.iloc[0] if len(gr) else None

    def main_author(a):
        a = re.split(r"[,(]", a.strip())[0]
        return re.sub(r"\s+", " ", a).strip()

    cands = {main_author(a) for a in vals if main_author(a)}
    return max(cands, key=lambda x: (len(x), x)) if cands else None


# ---------- publisher (prefer Metabooks > Goodreads > Amazon) ----------
def fuse_publisher(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        v = g.loc[g["source"].eq(src), "publisher"].dropna()
        if len(v):
            return str(v.iloc[0]).strip()
    return None


# ---------- publish_year (prefer Metabooks > Goodreads > Amazon; guard range) ----------
def fuse_publish_year(g: pd.DataFrame, year_min=1400, year_max=2100):
    for src in ["metabooks","goodreads","amazon"]:
        v = pd.to_numeric(g.loc[g["source"].eq(src), "publish_year"], errors="coerce").dropna()
        if len(v):
            y = int(v.iloc[0])
            if year_min <= y <= year_max:
                return y
    return None
# ---------- language (prefer Metabooks > Goodreads) ----------
def fuse_language(g: pd.DataFrame):
    """
    Prefer Metabooks language; if missing/NaN/empty, use Goodreads.
    Ignore Amazon entirely.
    """
    sub = g[g["source"].isin(["metabooks", "goodreads"])][["source", "language"]].copy()
    # normalize empties to NaN
    sub["language"] = sub["language"].astype(str).str.strip()
    sub.loc[sub["language"].str.lower().isin(["", "nan", "none"]), "language"] = np.nan

    # Metabooks first
    mb = sub.loc[sub["source"].eq("metabooks"), "language"].dropna()
    if len(mb):
        return mb.iloc[0]

    # Goodreads fallback
    gr = sub.loc[sub["source"].eq("goodreads"), "language"].dropna()
    if len(gr):
        return gr.iloc[0]

    return None


# ---------- rating (prefer Metabooks > Goodreads > Amazon) ----------
def fuse_rating(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        vv = pd.to_numeric(g.loc[g["source"].eq(src), "rating"], errors="coerce").dropna()
        if len(vv):
            return float(vv.iloc[0])
    return None


# ---------- numratings (keep it the same -> align with rating source; MB > GR > AZ) ----------
def fuse_numratings(g: pd.DataFrame):
    for src in ["metabooks","goodreads","amazon"]:
        vv = pd.to_numeric(g.loc[g["source"].eq(src), "numratings"], errors="coerce").dropna()
        if len(vv):
            return int(vv.iloc[0])
    return None


# ---------- genres (union of unique genres from GR + MB) ----------
def _parse_genres(s):
    if pd.isna(s): return []
    s = str(s).strip()
    try:
        lst = ast.literal_eval(s)
        if isinstance(lst, list): 
            return [str(x) for x in lst]
    except Exception:
        pass
    return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]

def fuse_genres(g: pd.DataFrame, as_list=False):
    sub = g[g["source"].isin(["goodreads","metabooks"])]
    all_genres = []
    for s in sub["genres"].dropna():
        all_genres.extend(_parse_genres(s))
    uniq = sorted({x.strip().title() for x in all_genres if x})
    return uniq if as_list else (", ".join(uniq) if uniq else None)


# ---------- page_count (take Metabooks first then Goodreads) ----------
def fuse_page_count(g: pd.DataFrame, min_pages=20):
    """
    Prefer Metabooks page_count; if missing, use Goodreads.
    Do NOT use Amazon (it doesn't provide page_count).
    """
    # Metabooks
    mb = pd.to_numeric(g.loc[g["source"].eq("metabooks"), "page_count"], errors="coerce").dropna()
    mb = mb[mb >= min_pages]
    if len(mb):
        return int(round(float(mb.iloc[0])))

    # Goodreads (fallback)
    gr = pd.to_numeric(g.loc[g["source"].eq("goodreads"), "page_count"], errors="coerce").dropna()
    gr = gr[gr >= min_pages]
    if len(gr):
        return int(round(float(gr.iloc[0])))

    return None


# ---------- price (prefer Metabooks > Goodreads ) ----------
def fuse_price(g: pd.DataFrame):
    """
    Prefer Metabooks price; if missing/NaN or non-positive, use Goodreads.
    Ignore Amazon entirely (column is artificial and NaN-filled).
    """
    sub = g[g["source"].isin(["metabooks", "goodreads"])][["source", "price"]].copy()
    sub["price"] = pd.to_numeric(sub["price"], errors="coerce")

    # Metabooks first
    mb = sub.loc[sub["source"].eq("metabooks"), "price"].dropna()
    mb = mb[mb > 0]
    if len(mb):
        return float(round(mb.iloc[0], 2))

    # Goodreads fallback
    gr = sub.loc[sub["source"].eq("goodreads"), "price"].dropna()
    gr = gr[gr > 0]
    if len(gr):
        return float(round(gr.iloc[0], 2))

    return None


# ---------- Main fusion pipeline ----------
def fuse_all(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Fuse datasets, keeping only ISBNs present in at least two sources."""
    long = build_longform(amazon, goodreads, metabooks)

    # ✅ Keep only ISBNs that appear in at least two different sources
    long = long[long.groupby("isbn_clean")["source"].transform("nunique") >= 2]

    fused = (long.groupby("isbn_clean", as_index=False)
                  .apply(lambda g: pd.Series({
                      "title":         fuse_title(g),
                      "author":        fuse_author(g),
                      "publisher":     fuse_publisher(g),
                      "publish_year":  fuse_publish_year(g),
                      "language":      fuse_language(g),
                      "rating":        fuse_rating(g),
                      "numratings":    fuse_numratings(g),
                      "genres":        fuse_genres(g, as_list=False),
                      "page_count":    fuse_page_count(g),
                      "price":         fuse_price(g),
                  }))
                  .reset_index(drop=True))

    # Dtype tidy-up (cast only when safe)
    for col, dtype in [("publish_year","Int16"), ("page_count","Int32"), ("numratings","Int64")]:
        fused[col] = pd.to_numeric(fused[col], errors="ignore").astype(dtype, errors="ignore")

    # Column order
    fused = fused[["isbn_clean","title","author","publisher","publish_year",
                   "rating","numratings","genres","page_count","price"]]
    return fused

In [ ]:
import pandas as pd, numpy as np, re, ast

# Amazon is missing these (in your setup):
AMAZON_MISSING = ["rating","numratings","genres","page_count","price"]

# Union schema we care about
BASE_COLS = [
    "id","title","author","publisher","publish_year",
    "rating","numratings","genres","page_count","price","isbn_clean"
]

def _add_missing_cols(df, cols):
    df = df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = np.nan
    return df

def build_longform(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Normalize inputs to the same schema, add source label, and stack them."""
    amz = _add_missing_cols(amazon.copy(), AMAZON_MISSING)
    gr  = goodreads.copy()
    mb  = metabooks.copy()

    amz["source"] = "amazon"
    gr["source"]  = "goodreads"
    mb["source"]  = "metabooks"

    def keep_cols(df): 
        return df[[c for c in BASE_COLS if c in df.columns] + ["source"]]
    
    long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)

    # normalize empties to NaN
    for c in ["title","author","publisher","genres","isbn_clean","id"]:
        if c in long.columns: 
            long[c] = long[c].replace({"": np.nan})
    return long


# ---------- title ----------
def fuse_title(g: pd.DataFrame):
    titles = g["title"].dropna().astype(str).tolist()
    if not titles: return None
    def clean_title(t):
        t = t.strip()
        t = re.sub(r"\s*[\(\[].*?[\)\]]\s*$", "", t)
        t = re.split(r"\s*[:\-]\s+", t)[0]
        return t.strip()
    pairs, dedup = [(t, clean_title(t)) for t in titles], {}
    for orig, c in pairs:
        key = c.lower()
        if key not in dedup: dedup[key] = orig
    best_key = min(dedup.keys(), key=lambda k: (len(k), len(k.split()), k))
    return dedup[best_key]

# ---------- author ----------
def fuse_author(g: pd.DataFrame):
    vals = g["author"].dropna().astype(str).tolist()
    if not vals: return None
    def main_author(a):
        a = re.split(r"[,(]", a.strip())[0]
        return re.sub(r"\s+", " ", a).strip()
    cands = {main_author(a) for a in vals if main_author(a)}
    return max(cands, key=lambda x: (len(x), x)) if cands else None

# ---------- publisher ----------
PUB_STOP = {"publishers","publisher","press","books","book","ltd","inc","co","corp",
            "limited","imprint","group","llc","the","and"}
def _norm_pub(p):
    p = re.sub(r"[\(\)\[\],.]", " ", str(p).lower())
    toks = [t for t in re.split(r"\s+", p) if t and t not in PUB_STOP]
    return " ".join(toks)

def fuse_publisher(g: pd.DataFrame):
    pubs = g["publisher"].dropna().astype(str)
    if pubs.empty: return None
    roots = pubs.map(_norm_pub)
    vc = roots.value_counts()
    maxc = vc.iloc[0]
    cands = [r for r,c in vc.items() if c == maxc]
    root = max(cands, key=lambda r: (len(r), r))
    return " ".join(w.capitalize() for w in root.split())

# ---------- publish_year ----------
def fuse_publish_year(g: pd.DataFrame, year_min=1400, year_max=2100):
    yrs = pd.to_numeric(g["publish_year"], errors="coerce").dropna().astype(int)
    yrs = yrs[(yrs>=year_min) & (yrs<=year_max)]
    return int(yrs.min()) if len(yrs) else None

# ---------- rating (GR + MB only; 10–20% proximity rule) ----------
def fuse_rating(g: pd.DataFrame, delta=0.15):
    sub = g[g["source"].isin(["goodreads","metabooks"])][["source","rating","numratings"]].copy()
    sub["rating"] = pd.to_numeric(sub["rating"], errors="coerce")
    sub["numratings"] = pd.to_numeric(sub["numratings"], errors="coerce")
    sub = sub.dropna(subset=["rating"])

    gr = sub[sub["source"].eq("goodreads")]
    mb = sub[sub["source"].eq("metabooks")]

    if len(gr) and len(mb):
        r1, n1 = float(gr.iloc[0]["rating"]), float(gr.iloc[0]["numratings"] or 0)
        r2, n2 = float(mb.iloc[0]["rating"]), float(mb.iloc[0]["numratings"] or 0)
        base = max(n1, n2)
        if base == 0: return (r1 + r2) / 2
        diff = abs(n1 - n2) / base
        return (r1*n1 + r2*n2) / (n1 + n2) if diff <= delta else (r1 if n1 >= n2 else r2)

    if len(gr): return float(gr.iloc[0]["rating"])
    if len(mb): return float(mb.iloc[0]["rating"])
    return None

# ---------- numratings (consistent with rating decision) ----------
def fuse_numratings(g: pd.DataFrame, delta=0.15):
    sub = g[g["source"].isin(["goodreads","metabooks"])][["source","rating","numratings"]].copy()
    sub["numratings"] = pd.to_numeric(sub["numratings"], errors="coerce")

    gr = sub[sub["source"].eq("goodreads")].dropna(subset=["numratings"])
    mb = sub[sub["source"].eq("metabooks")].dropna(subset=["numratings"])

    if len(gr) and len(mb):
        n1, n2 = float(gr.iloc[0]["numratings"] or 0), float(mb.iloc[0]["numratings"] or 0)
        base = max(n1, n2)
        if base == 0: return int(round((n1 + n2) / 2))
        diff = abs(n1 - n2) / base
        return int(n1 + n2) if diff <= delta else int(max(n1, n2))

    if len(gr): return int(gr.iloc[0]["numratings"])
    if len(mb): return int(mb.iloc[0]["numratings"])
    return None

# ---------- genres (union of GR + MB) ----------
def _parse_genres(s):
    if pd.isna(s): return []
    s = str(s).strip()
    try:
        lst = ast.literal_eval(s)
        if isinstance(lst, list): return [str(x) for x in lst]
    except Exception:
        pass
    return [p.strip() for p in re.split(r"[|,;/]", s) if p.strip()]

def fuse_genres(g: pd.DataFrame, as_list=False):
    sub = g[g["source"].isin(["goodreads","metabooks"])]
    all_genres = []
    for s in sub["genres"].dropna():
        all_genres.extend(_parse_genres(s))
    uniq = sorted({x.strip().title() for x in all_genres if x})
    return uniq if as_list else (", ".join(uniq) if uniq else None)

# ---------- page_count (GR + MB only; mean if close, else prefer GR by default) ----------
def fuse_page_count(g: pd.DataFrame, min_pages=20, rel_tol=0.05, prefer_on_disagreement="goodreads"):
    sub = g[g["source"].isin(["goodreads","metabooks"])].copy()
    sub["page_count"] = pd.to_numeric(sub["page_count"], errors="coerce")
    vals = sub["page_count"].dropna()
    vals = vals[vals >= min_pages]
    if not len(vals): return None
    if len(vals) == 1: return int(round(float(vals.iloc[0])))

    pmin, pmax = float(vals.min()), float(vals.max())
    if pmax != 0 and (pmax - pmin) / pmax <= rel_tol:
        return int(round(vals.mean()))

    pref = prefer_on_disagreement.lower()
    if pref in ("goodreads","metabooks"):
        chosen = sub.loc[sub["source"].eq(pref), "page_count"].dropna()
        if len(chosen): return int(round(float(chosen.iloc[0])))
        other = "metabooks" if pref=="goodreads" else "goodreads"
        chosen = sub.loc[sub["source"].eq(other), "page_count"].dropna()
        if len(chosen): return int(round(float(chosen.iloc[0])))
    return int(round(pmax))  # final guard

# ---------- price (prefer Amazon, else Goodreads, else None) ----------
def fuse_price(g):
    """
    Prefer Amazon price if available and >0; else Goodreads if >0; else None.
    """
    sub = g[["source","price"]].copy()
    sub["price"] = pd.to_numeric(sub["price"], errors="coerce")
    sub = sub[(sub["price"].notna()) & (sub["price"] > 0)]
    if sub.empty:
        return None

    az = sub[sub["source"].str.lower().eq("amazon")]
    if len(az):
        return float(round(az.iloc[0]["price"], 2))

    gr = sub[sub["source"].str.lower().eq("goodreads")]
    if len(gr):
        return float(round(gr.iloc[0]["price"], 2))

    return None  # add a Metabooks block here if you want a fallback



# ---------- Main fusion pipeline ----------
def fuse_all(amazon: pd.DataFrame, goodreads: pd.DataFrame, metabooks: pd.DataFrame) -> pd.DataFrame:
    """Fuse datasets, keeping only ISBNs present in at least two sources."""
    long = build_longform(amazon, goodreads, metabooks)

    # ✅ Keep only ISBNs that appear in at least two different sources
    long = long[long.groupby("isbn_clean")["source"].transform("nunique") >= 2]

    fused = (long.groupby("isbn_clean", as_index=False)
                  .apply(lambda g: pd.Series({
                      "title":         fuse_title(g),
                      "author":        fuse_author(g),
                      "publisher":     fuse_publisher(g),
                      "publish_year":  fuse_publish_year(g),
                      "rating":        fuse_rating(g, delta=0.15),
                      "numratings":    fuse_numratings(g, delta=0.15),
                      "genres":        fuse_genres(g, as_list=False),
                      "page_count":    fuse_page_count(g, rel_tol=0.05, prefer_on_disagreement="goodreads"),
                      "price":         fuse_price(g),
                  }))
                  .reset_index(drop=True))

    # Dtype tidy-up (cast only when safe)
    for col, dtype in [("publish_year","Int16"), ("page_count","Int32"), ("numratings","Int64")]:
        fused[col] = pd.to_numeric(fused[col], errors="ignore").astype(dtype, errors="ignore")

    # Column order
    fused = fused[["isbn_clean","title","author","publisher","publish_year",
                   "rating","numratings","genres","page_count","price"]]
    return fused


In [22]:
fused_df = fuse_all(amazon, goodreads, metabooks)

/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_83383/1325584272.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  long = pd.concat([keep_cols(amz), keep_cols(gr), keep_cols(mb)], ignore_index=True)
/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_83383/1325584272.py:192: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({
/var/folders/99/1mpdqqd52gx9ngr3lnl8wkzh0000gn/T/ipykernel_83383/1325584272.py:207: Futu

In [24]:
fused_df.shape

(38828, 10)

In [25]:
fused_df.isnull().sum()

isbn_clean         0
title              0
author             0
publisher          0
publish_year     120
rating             0
numratings         0
genres          1380
page_count      3417
price           2388
dtype: int64

In [39]:
fused_df.to_parquet("ml-datasets/golden_fused_books.parquet", index=False)